In [1]:
import pytensor
import pytensor.tensor as at


In [2]:
k = at.iscalar("k")
A = at.vector("A")

# Symbolic description of the result
result, updates = pytensor.scan(fn=lambda prior_result, A: prior_result*2 ,
                              outputs_info=at.ones_like(A),
                              sequences=A,
                              n_steps=k)




ValueError: When compiling the inner function of scan (the function called by scan in each of its iterations) the following error has been encountered: The initial state (`outputs_info` in scan nomenclature) of variable IncSubtensor{Set;:int64:}.0 (argument number 1) has 2 dimension(s), while the corresponding variable in the result of the inner function of scan (`fn`) has 0 dimension(s) (it should be one less than the initial state). For example, if the inner function of scan returns a vector of size d and scan uses the values of the previous time-step, then the initial state in scan should be a matrix of shape (1, d). The first dimension of this matrix corresponds to the number of previous time-steps that scan uses in each of its iterations. In order to solve this issue if the two varialbe currently have the same dimensionality, you can increase the dimensionality of the variable in the initial state of scan by using dimshuffle or shape_padleft. 

In [33]:
# compiled function that returns A**k
power = pytensor.function(inputs=[A,k], outputs=result, updates=updates)

an = power(range(10),2)
an[-1,...]

array([4., 4., 4., 4., 4., 4., 4., 4., 4., 4.])

In [6]:
from pytensor.scan.utils import until

In [19]:
def power_of_2(previous, previous_power, max_value):
    return [previous+1, previous_power+2], until(previous_power > max_value)


In [20]:
max_value = at.scalar()
values, _ = pytensor.scan(power_of_2,
                        outputs_info = [at.constant(1.), at.constant(1.)],
                        non_sequences = max_value,
                        n_steps = 10)


In [21]:
f = pytensor.function([max_value], values)

print(f(8))

[array([2., 3., 4., 5., 6.], dtype=float32), array([ 3.,  5.,  7.,  9., 11.], dtype=float32)]


In [5]:
import numpy

coefficients = pytensor.tensor.vector("coefficients")
x = at.scalar("x")

max_coefficients_supported = 10000

# Generate the components of the polynomial
components, updates = pytensor.scan(fn=lambda coefficient, power, free_variable: (coefficient * (free_variable ** power)).sum(),
                                  outputs_info=None,
                                  sequences=[coefficients, pytensor.tensor.arange(max_coefficients_supported)],
                                  non_sequences=x)
# Sum them up
polynomial = components.sum()


In [6]:

# Compile a function
calculate_polynomial = pytensor.function(inputs=[coefficients, x], outputs=polynomial)

# Test
test_coefficients = numpy.asarray([1, 0, 2], dtype=numpy.float32)
test_value = 3
print(calculate_polynomial(test_coefficients, test_value))
print(1.0 * (3 ** 0) + 0.0 * (3 ** 1) + 2.0 * (3 ** 2))

19.0
19.0
